# 13 DBSCAN

依赖安装说明：`pip install numpy matplotlib scikit-learn`

DBSCAN 是基于密度的聚类算法。它不需要预先指定簇数，并且可以识别噪声点。


## 0. 学习目标和阅读地图

DBSCAN 的重点是“密度连通”。你需要掌握：

1. 核心点、边界点、噪声点的区别。
2. `eps` 和 `min_samples` 如何决定密度标准。
3. 为什么 DBSCAN 能发现非球形簇。
4. 为什么它难处理不同密度的簇。


## 1. 数学逻辑

DBSCAN 有两个关键参数：

- `eps`：邻域半径。
- `min_samples`：成为核心点需要的最少邻居数。

如果一个点的 eps 邻域内至少有 `min_samples` 个点，它就是核心点。簇从核心点开始，把密度可达的点连起来。无法归入任何簇的点标记为噪声 `-1`。


## 1.1 推导拆开看：密度可达

如果一个点在半径 `eps` 内有足够多邻居，它是核心点。核心点可以把周围点拉进同一个簇。

两个点属于同一簇，不一定要彼此很近；只要它们能通过一串核心点连接起来，就是密度可达。

这就是 DBSCAN 能识别月牙形簇的原因：它不需要簇围绕一个中心，只需要密度区域连通。


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import StandardScaler

np.random.seed(42)
X, _ = make_moons(n_samples=260, noise=0.08, random_state=42)
noise = np.random.uniform(low=[-1.8, -1.0], high=[2.8, 1.6], size=(25, 2))
X = np.vstack([X, noise])
X_s = StandardScaler().fit_transform(X)


## 1.2 为什么要标准化

DBSCAN 同样依赖距离。`eps=0.25` 的含义完全取决于特征尺度。如果一个特征范围很大，它会主导邻域判断。

所以代码先用 `StandardScaler`，再在标准化后的空间里定义 `eps`。


In [ ]:
# 从零实现：简化版 DBSCAN，重点看核心逻辑

def region_query(X, i, eps):
    d = np.sqrt(((X - X[i]) ** 2).sum(axis=1))
    return np.where(d <= eps)[0]

def dbscan_simple(X, eps=0.25, min_samples=5):
    labels = np.full(len(X), -99)  # -99 表示未访问，-1 表示噪声
    cluster_id = 0
    for i in range(len(X)):
        if labels[i] != -99:
            continue
        neighbors = list(region_query(X, i, eps))
        if len(neighbors) < min_samples:
            labels[i] = -1
            continue
        labels[i] = cluster_id
        queue = neighbors[:]
        while queue:
            j = queue.pop(0)
            if labels[j] == -1:
                labels[j] = cluster_id
            if labels[j] != -99:
                continue
            labels[j] = cluster_id
            j_neighbors = list(region_query(X, j, eps))
            if len(j_neighbors) >= min_samples:
                queue.extend(j_neighbors)
        cluster_id += 1
    return labels

labels_simple = dbscan_simple(X_s, eps=0.25, min_samples=5)
print('从零 DBSCAN 标签:', sorted(set(labels_simple)))


## 1.3 从零实现代码怎么读

简化版 DBSCAN 的核心是队列扩展：

1. 找到一个未访问点。
2. 如果邻居不够，暂时标成噪声。
3. 如果它是核心点，就创建新簇。
4. 把它的邻居放入队列，继续扩展密度可达区域。

这和图搜索很像：核心点之间通过邻域连接起来。


In [ ]:
model = DBSCAN(eps=0.25, min_samples=5)
labels = model.fit_predict(X_s)
print('sklearn DBSCAN 标签:', sorted(set(labels)))
print('噪声点数量:', np.sum(labels == -1))

plt.scatter(X_s[:,0], X_s[:,1], c=labels, cmap='tab10', s=24)
plt.title('DBSCAN 可以发现非球形簇并标记噪声')
plt.show()


In [ ]:
# 诊断：eps 改变时，簇数量和噪声数量如何变化
eps_values = np.linspace(0.12, 0.45, 10)
for eps_value in eps_values:
    lab = DBSCAN(eps=eps_value, min_samples=5).fit_predict(X_s)
    n_clusters = len(set(lab)) - (1 if -1 in lab else 0)
    n_noise = np.sum(lab == -1)
    print(f'eps={eps_value:.2f} | clusters={n_clusters} | noise={n_noise}')


## 2.1 如何诊断 DBSCAN

`eps` 是最敏感参数。太小会把很多点标成噪声；太大会把本该分开的簇连起来。

一个实用方法是看 k-distance 曲线：对每个点计算第 `min_samples` 近邻距离，寻找曲线的拐点作为 eps 候选。


## 2. 常见误区

- `eps` 很敏感，太小会产生大量噪声，太大会把簇连在一起。
- 特征尺度会影响距离，通常需要标准化。
- 不同密度的簇会让 DBSCAN 很难同时处理好。

## 3. 小实验

- 改 `eps`，观察簇数量和噪声点数量。
- 改 `min_samples`，观察核心点条件变化。
- 对未标准化数据直接跑，观察差异。


## 5. 复习清单

- DBSCAN 不需要指定簇数。
- 它能标记噪声点。
- 它基于密度连通，适合非球形簇。
- 不同密度簇和高维数据会让参数选择变难。
